# Nutrition5k: experimentos híbridos móviles

Híbridos móviles: convolución en las primeras etapas y atención en las últimas, pensados para el móvil. Cada uno tiene
un gemelo sin atención, así que cada pareja aísla el efecto de la atención (Q2).

| Clave | Modelo | Pareja |
|---|---|---|
| `fastvit_sa12` | FastViT-SA12: RepMixer en las etapas 1-3, autoatención en la 4 | Q2b, con `fastvit_s12` |
| `fastvit_s12` | FastViT-S12: RepMixer en las cuatro etapas (sin atención) | Q2b |
| `v4hm` | MobileNetV4-Hybrid-M: `v4m` con bloques de atención al final (misma receta `e500_r224`) | Q2a, con `v4m` (en `mobilenet/`) |

- Notebook fino: solo llama a `python -m nutrition5k ...`. Toda la lógica está en `src/nutrition5k/`; los backbones
  de esta familia, en `src/nutrition5k/models/hybrid_mobile.py`.
- Los runs se guardan en `outputs/hybrid_mobile/<full|pilot>/<backbone>/<run>/`: `config.json` (hiperparámetros y entorno),
  `history.csv` (una fila por época), `best.pt`, `last.pt`, predicciones y métricas de validación y test.
- Funciona en local y en un pod de RunPod. En RunPod, `outputs/` se enlaza al volumen (`/workspace/nutrition5k`).
- Volver a ejecutar una celda es seguro: los runs terminados se saltan y los interrumpidos se reanudan desde `last.pt`.
- Si existe `WANDB_API_KEY`, cada run se refleja en wandb.ai (proyecto `nutrition5k`).
- Experimentos finales (rama `final-experiments`, imágenes cenitales). Etapa 1: cabezas de regresión con el
  backbone congelado, sobre features guardadas, en todos los backbones. Etapa 2: profundidad del fine-tuning con
  la cabeza elegida. Etapa 3: semillas, comprobación de la cabeza y regularización en los finalistas. Las etapas
  anteriores (frames laterales) están en el historial de git (commit `13e0b27`) y en las notas.
- Receta común: dropout y drop path a 0 en todos los backbones, mismo recorte y preprocesado, BatchNorm congelada.
- Las decisiones se toman con **validación** en `results.ipynb` de esta carpeta, nunca con test.

## Parámetros

In [ ]:
import os

FULL = True          # False: piloto 32/8 para pruebas rápidas
EPOCHS = 80
BATCH = 32
WORKERS = min(15, os.cpu_count()-2)  # procesos que decodifican las imágenes

FULL_FLAG = "--full" if FULL else ""
SPLIT = "full" if FULL else "pilot"

## Preparación del entorno

In [ ]:
import os
import sys
from pathlib import Path

# The volume is mounted at /workspace only on RunPod.
RUNPOD = Path("/workspace").is_dir()
if RUNPOD:
    # SSH sessions (VS Code Remote) do not inherit the pod's variables, such as the W&B secret.
    for variable in Path("/proc/1/environ").read_text().split("\0"):
        name, _, value = variable.partition("=")
        if name == "WANDB_API_KEY":
            os.environ.setdefault(name, value)
    # The template points the Hugging Face cache at the volume, where its symlinks break
    # (object storage); keep the pretrained weights on the pod's local disk instead.
    os.environ["HF_HOME"] = "/root/.cache/huggingface"

REPOSITORY_URL = "https://github.com/victorabellotorres/thesis-multimodal-health-ai.git"
# Local disk of the pod: fast, but lost when the pod ends.
REPOSITORY_DIR = Path("/root/thesis-multimodal-health-ai")
BRANCH = "final-experiments"
# Persistent storage: the volume mounted at /workspace.
PERSIST_ROOT = Path("/workspace/nutrition5k")

if RUNPOD:
    if (REPOSITORY_DIR / ".git").is_dir():
        !git -C {REPOSITORY_DIR} fetch
        !git -C {REPOSITORY_DIR} checkout {BRANCH}
        !git -C {REPOSITORY_DIR} pull --ff-only
    else:
        !git clone -b {BRANCH} {REPOSITORY_URL} {REPOSITORY_DIR}
    %cd {REPOSITORY_DIR}
    # The image's system Python is "externally managed" (PEP 668); the pod is disposable.
    %pip install -q --break-system-packages -e .[train]
elif Path.cwd().parent.name == "notebooks":
    # Local: the package is already installed in code/.venv; run from the repository root.
    os.chdir("../..")

# The kernel's interpreter, so `!` commands use the same environment locally (VS Code may not activate .venv).
PYTHON = sys.executable

if RUNPOD:
    # Results live on the volume so nothing is lost when the pod ends.
    (PERSIST_ROOT / "outputs").mkdir(parents=True, exist_ok=True)
    if not Path("outputs").exists():
        Path("outputs").symlink_to(PERSIST_ROOT / "outputs")
print("W&B:", "on" if os.environ.get("WANDB_API_KEY") else "off (only local files)")

En local, `fetch` y `prepare` trabajan directamente sobre `data/` y se saltan lo que ya está descargado.

En RunPod, si el `.tar` ya existe en el volumen se restaura; si no, `prepare` descarga la imagen cenital de cada plato (`rgb.png` y `depth_raw.png`, unos 2 GB en total) y se guarda el `.tar` en el volumen para el próximo pod. Lo normal es subir el `.tar` preparado en el portátil (ver el README), así la GPU no espera a las descargas.

Si `prepare` falla (por ejemplo, un timeout que persiste tras los reintentos), no se crea el `.tar`; volver a ejecutar la celda continúa donde se quedó mientras el pod siga encendido.

In [ ]:
DATA_TAR = PERSIST_ROOT / f"data-{SPLIT}-overhead.tar"

if RUNPOD and DATA_TAR.is_file():
    !tar -xf {DATA_TAR}
else:
    !{PYTHON} -m nutrition5k fetch
    !{PYTHON} -m nutrition5k prepare {FULL_FLAG}
    if RUNPOD and _exit_code == 0:  # a failed prepare must not be cached as a partial .tar
        !tar -cf {DATA_TAR} data
!{PYTHON} -m nutrition5k preprocess  # excluded_dishes.csv and dish_groups.csv
!{PYTHON} -m nutrition5k check {FULL_FLAG}

## Baseline (media del conjunto de entrenamiento)

In [ ]:
!{PYTHON} -m nutrition5k baseline {FULL_FLAG}

In [ ]:
def train(backbone, unfreeze="all", heads="single", hidden=(512,), image_size=224, backbone_lr=1e-4, seed=0,
          head_width=256, pooling=None, augment="flip", all_seeds=False):
    """Run one configuration; finished runs are skipped, interrupted ones resume."""
    hidden_args = " ".join(f"--hidden {size}" for size in (hidden or (0,)))
    command = (
        f"{PYTHON} -m nutrition5k train --backbone {backbone} --unfreeze {unfreeze} "
        f"--heads {heads} {hidden_args} --image-size {image_size} --backbone-lr {backbone_lr} "
        f"--head-width {head_width} --augment {augment} "
        f"--epochs {EPOCHS} --batch-size {BATCH} --num-workers {WORKERS} --seed {seed} {FULL_FLAG}"
    )
    if pooling:
        command += f" --pooling {pooling}"
    if all_seeds:
        command += " --all-seeds"
    print(command)
    get_ipython().system(command)


def heads(backbone, grid=False, heads="single", hidden=(512,), head_width=256, pooling=None, augment="flip",
          seed=0):
    """Stage 1: heads on the frozen backbone. Features are extracted once and cached; finished heads are skipped."""
    hidden_args = " ".join(f"--hidden {size}" for size in (hidden or (0,)))
    command = f"{PYTHON} -m nutrition5k heads --backbone {backbone} --num-workers {WORKERS} --seed {seed} {FULL_FLAG}"
    if grid:
        command += " --grid"
    else:
        command += f" --heads {heads} {hidden_args} --head-width {head_width} --augment {augment}"
        if pooling:
            command += f" --pooling {pooling}"
    print(command)
    get_ipython().system(command)

## Etapa 1: cabezas de regresión con el backbone congelado

Cada backbone pasa una sola vez por todas las imágenes (con sus 8 simetrías: giros de 90° y volteo) y sus features se
guardan en `outputs/hybrid_mobile/<full|pilot>/heads/<backbone>/features.pt`. Después cada cabeza se entrena en segundos
sobre esas features: semilla 0, 80 épocas, lr de la cabeza 1e-3. El resultado también es el `unfreeze 0` de la etapa 2.

1. Rejilla reducida (`--grid`, 9 cabezas): tronco {0, 512, 1024×1024} × cabeza {lineal, oculta de 256} ×
   agrupación {single, grouped}. Con cabeza lineal, `grouped` es igual que `single` y no se repite.
2. Aumentación D4 (giros de 90° y volteos, solo en entrenamiento) frente al volteo horizontal, en la cabeza por defecto.
3. Ruido: semillas 1 y 2 de la cabeza por defecto (`single`, tronco 512, cabeza 256), para estimar el umbral de empate.

Si la cabeza elegida en `results.ipynb` no es la de por defecto, repetir 2 (y el pooling, en ViT) sobre ella: son
segundos con las features ya guardadas.

In [ ]:
BACKBONES = ('fastvit_s12', 'fastvit_sa12', 'v4hm')

for backbone in BACKBONES:
    heads(backbone, grid=True)
    heads(backbone, augment="d4")
    for seed in (1, 2):
        heads(backbone, seed=seed)

## Etapa 2: profundidad del fine-tuning (pendiente)

Con la cabeza fijada en la etapa 1: `unfreeze` 2 y `all` en todos los backbones (el 0 sale de la etapa 1), lr del
backbone 1e-4. Se rellena cuando la etapa 1 esté decidida.